In [ ]:
import base64, json, os, shutil, subprocess, sys
from pathlib import Path

JOB_MANIFEST = json.loads(base64.b64decode("__JOB_MANIFEST_B64__"))
print({k: (v if k != "prompt" else v[:180]) for k, v in JOB_MANIFEST.items()})
assert int(JOB_MANIFEST["frame_num"]) >= 5 and (int(JOB_MANIFEST["frame_num"]) - 1) % 4 == 0
assert int(JOB_MANIFEST["sample_steps"]) >= 1
SCENES = JOB_MANIFEST["scenes"]
assert 1 <= len(SCENES) <= 4, "Worker supports 1-4 scenes per GPU run."


In [ ]:
WORK = Path("/kaggle/working")
WAN_DIR = WORK / "Wan2.1"
MODEL_DIR = WORK / "Wan2.1-T2V-1.3B"
OUT_DIR = WORK / "wan_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Wan-Video/Wan2.1.git", str(WAN_DIR)], check=True)
req = WORK / "requirements_no_flash.txt"
lines = (WAN_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
req.write_text("\n".join(line for line in lines if "flash_attn" not in line.lower() and "flash-attn" not in line.lower()) + "\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord", "ftfy", "dashscope", "huggingface_hub"], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU is not enabled for this kernel.")
print("GPU:", torch.cuda.get_device_name(0), "CUDA:", torch.version.cuda)


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="Wan-AI/Wan2.1-T2V-1.3B", local_dir=str(MODEL_DIR))
model_py = WAN_DIR / "wan/modules/model.py"
source = model_py.read_text(encoding="utf-8")
source = source.replace("from .attention import flash_attention", "from .attention import attention")
source = source.replace("flash_attention(", "attention(")
if "from .attention import attention" not in source or "flash_attention(" in source:
    raise RuntimeError("Wan T4 SDPA fallback patch verification failed.")
model_py.write_text(source, encoding="utf-8")
print("Wan T4 SDPA fallback patch applied.")


In [ ]:
import re

def safe_scene_id(value, index):
    value = str(value or f"scene_{index:02d}").strip().lower()
    value = re.sub(r"[^a-z0-9_-]+", "_", value).strip("_-")
    return (value or f"scene_{index:02d}")[:48]

reports = []
used_names = set()
for index, scene in enumerate(SCENES, start=1):
    prompt = str(scene.get("prompt", "")).strip()
    if not prompt:
        raise ValueError(f"Scene {index} has an empty prompt.")
    scene_id = safe_scene_id(scene.get("id"), index)
    if scene_id in used_names:
        raise ValueError(f"Duplicate scene id after normalization: {scene_id}")
    used_names.add(scene_id)
    output_path = OUT_DIR / f"{index:02d}_{scene_id}.mp4"
    command = [
        sys.executable, "generate.py", "--task", "t2v-1.3B",
        "--size", str(JOB_MANIFEST["size"]),
        "--frame_num", str(JOB_MANIFEST["frame_num"]),
        "--sample_steps", str(JOB_MANIFEST["sample_steps"]),
        "--ckpt_dir", str(MODEL_DIR), "--offload_model", "True",
        "--t5_cpu", "--sample_shift", "8", "--sample_guide_scale", "5",
        "--save_file", str(output_path), "--prompt", prompt,
    ]
    print(f"Generating scene {index}/{len(SCENES)}: {scene_id}")
    subprocess.run(command, cwd=str(WAN_DIR), check=True)
    if not output_path.is_file() or output_path.stat().st_size < 50_000:
        raise RuntimeError(f"Scene output missing or too small: {output_path.name}")
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries",
         "format=duration:stream=codec_type,width,height", "-of", "json",
         str(output_path)], capture_output=True, text=True, check=True
    )
    data = json.loads(probe.stdout)
    video = next((item for item in data.get("streams", [])
                  if item.get("codec_type") == "video"), None)
    duration = float((data.get("format") or {}).get("duration") or 0)
    if not video or duration <= 0:
        raise RuntimeError(f"FFprobe could not validate {output_path.name}")
    reports.append({
        "scene_id": scene_id, "title": str(scene.get("title", scene_id)),
        "prompt": prompt, "file": output_path.name,
        "bytes": output_path.stat().st_size, "duration_seconds": duration,
        "width": video.get("width"), "height": video.get("height"),
        "ai_generated": True,
    })
    print("VALIDATED:", reports[-1])
(OUT_DIR / "scene_manifest.json").write_text(
    json.dumps({"scenes": reports}, indent=2), encoding="utf-8"
)


In [ ]:
print("Worker outputs:")
for item in sorted(OUT_DIR.iterdir()):
    if item.is_file(): print(item.name, item.stat().st_size)
assert list(OUT_DIR.glob("*.mp4")), "No MP4 was generated."
assert len(list(OUT_DIR.glob("*.mp4"))) == len(SCENES), "Scene output count mismatch."


In [ ]:
# Keep Kaggle's published kernel output small: retain only the deliverables.
# Wan2.1 source and model weights are temporary inference dependencies, not artifacts.
for temporary_path in (WAN_DIR, MODEL_DIR):
    if temporary_path.exists():
        print(f"Removing temporary inference directory: {temporary_path}")
        shutil.rmtree(temporary_path, ignore_errors=True)
req.unlink(missing_ok=True)
deliverables = sorted(OUT_DIR.glob("*.mp4"))
if not deliverables:
    raise RuntimeError("No generated MP4 remains in the deliverables directory.")
print("Final Kaggle deliverables:")
for item in deliverables:
    print(item.name, item.stat().st_size)
print("Temporary model and source directories removed.")
